# TME UE IAR: From Traditional EA to Quality-Diversity algorithms

Version du 13/04/2026

* Etudiant1: Nom prénom
* Etudiant2: Nom prénom

## 1. Introduction

### 1.1 Dépendances


In [ ]:
!pip install deap
!pip install matplotlib
!pip install numpy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random

### 1.2 Environnement utilisé pour les expériences

Vous allez utiliser un environnement simple pour permettre aux calculs d'être suffisamment rapides quelque soit votre environnement de travail. Il s'agit de la simulation d'un bras robotique articulé. Les paramètres explorés par l'algorithme évolutionnaire correspondent à l'angle que fait chaque segment par rapport au segment précédent. Il n'y a donc pas de politique à proprement parler.

La cellule suivante permet d'instancier l'environnement. La méthode `fw_kinematics(angles)` prend en entrée une liste d'angles relatifs (ce qui correspondra au génome de nos individus) et donne en sortie la position de l'effecteur terminal, ainsi qu'un booléen indiquant s'il y a une collision entre un segment du bras et les obstacles de l'environnement.

In [ ]:
import arm

lengths = [1.5] * 10  # Longueur des segments du bras
target_pos = [-3, 0.]  # Position de la cible
walls = np.array([
    [[-1, -2], [-1, 2]],
    [[-6.5, -2], [-1, -2]],
    [[-6.5, 2], [-1, 2]],
])  # Obstacles à éviter

# Environnement = bras articulé (vous pourrez garder ces mêmes paramètres pour vos expériences)
env = arm.Arm(
    lengths=lengths,
    walls=walls,
    target_pos=target_pos
)

# Test de l'environnement
angles = [0.1] * 10  # Angles du bras

pos_end_effector, collision = env.fw_kinematics(angles)
print(f"Position de l'effecteur : {pos_end_effector}")
print(f"Collision : {collision}")

# Affichage du bras
env.display_configuration()

L'objectif des expériences est de déterminer les configurations du bras permettant d'atteindre diverses positions de l'espace avec l'effecteur terminal. 
- La première partie du TP vise à atteindre la cible en rouge. 
- La deuxième partie du TP vise à trouver des configurations du bras permettant d'atteindre l'ensemble des positions atteignables, avec aussi peu de variance que possible entre les angles relatifs.

## 2. Atteindre une position cible avec trois variantes: FIT, NS et FIT+NS

Cette première partie vise à vous faire prendre en main 3 variantes d'apprentissage direct:
- FIT: algorithme élitiste guidé par une fitness globale
- NS: algorithme de recherche de nouveauté (novelty search)
- FIT+NS: algorithme combinant FIT et NS avec une approche multi-objectif 

Les trois sous-parties suivantes vous guident dans l'implémentation des trois variantes.

### 2.1. FIT: Optimisation mono-objectif de la fitness

Dans cette première variante, vous allez réutiliser l'algorithme `ea_elitist` du premier TME1, afin d'optimiser la configuration des articulations du bras robot dans le but de se rapprocher du point cible. 

Pour commencer, il faut définir une fonction d'évaluation d'une configuration angulaire (le génotype) du bras robot. La fonction doit retourner la distance de l'effecteur terminal du bras au point cible. Si une collision est détectée, la fonction doit à la place retourner `20`. Attention, dans deap les fonctions d'évaluation doivent toujours retourner des tuples, même dans le cas mono-objectif, il faut une valeur de retour du type `(fitness, )`.

#### Question 1.

En vous inspirant du code fourni dans l'introduction du TME, complétez la fonction d'évaluation suivante:

In [ ]:
def eval_arm(ind):
    """
    Fonction d'évaluation de l'algorithme évolutionnaire sur l'environnement de bras articulé.
    """

    # Initialisation de l'environnement
    env = arm.Arm(
        lengths=lengths,
        walls=walls,
        target_pos=target_pos
    )

    # A compléter
    ...

    return fitness,

In [ ]:
# Test de la fonction d'évaluation

print(eval_arm([0.1] * 10))  # Doit afficher (np.float(9.588679832249491),)
print(eval_arm([0.2] * 10))  # Doit afficher (20,)

L'implémentation de l'algorithme élitiste du TME1 vous est donnée ci-dessous. La première variante à tester consiste à simplement l'appliquer afin d'optimiser directement la fitness. Il n'y a rien à compléter pour l'instant dans les cellules suivantes.

Pour commencer, on définit des classes `FitnessMin` et `IndividualMin` car la tâche est de minimiser la distance à l'objectif.

In [ ]:
from deap import base, tools, creator

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("IndividualMin", list, fitness=creator.FitnessMin)

Les hyperparamètres donnés ci-dessous peuvent être conservés pour tout le TME.

In [ ]:
IND_SIZE=10
MIN_V=-np.pi
MAX_V=np.pi
CXPB=0.5
MUTPB=0.2

On instancie ensuite une toolbox dans laquelle on place des fonctions qu'on utilisera dans tout le TME.

In [ ]:
toolbox = base.Toolbox()
toolbox.register('attribute', random.uniform, MIN_V, MAX_V)
toolbox.register("mate", tools.cxSimulatedBinaryBounded,
                eta=15, low=MIN_V, up=MAX_V)
toolbox.register("mutate", tools.mutPolynomialBounded,
                eta=15, low=MIN_V, up=MAX_V, indpb=1/IND_SIZE)

La cellule suivante contient la définition de l'algorithme `ea_elistist`. Cet algorithme est 

In [ ]:
def ea_elitist(n, nbgen, evaluate):
    """
    Algorithme evolutionniste avec sélection élitiste
    :param n: taille de la population
    :param nbgen: nombre de generation
    :param evaluate: la fonction d'évaluation
    """
    random.seed()

    # Wrapper pour directement mettre la valeur de fitness dans l'individu après évaluation
    def wrap_evaluate(individual):
        individual.fitness.values = evaluate(individual)

    # Enregistrement des fonctions de création d'individus, de population, de sélection et d'évaluation dans le toolbox
    toolbox.register("individual", tools.initRepeat, creator.IndividualMin,
                    toolbox.attribute, n=IND_SIZE)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("select", tools.selBest)
    toolbox.register("evaluate", wrap_evaluate)

    # Enregistrement des statistiques pour suivre l'évolution de la population
    stats = tools.Statistics(key=lambda ind: ind.fitness.values)
    stats.register("min", np.min)

    # Enregistrement de l'historique de l'évolution de la population et de la meilleure solution trouvée
    logbook = tools.Logbook()
    hof = tools.HallOfFame(1)

    # Initialisation de la population
    population = toolbox.population(n=n)

    # =================================================================================================
    # PARTIE 3.3: sera à compléter plus tard, pour l'instant vous pouvez laisser la variable à None
    # Initialisation de l'archive structurée pour étudier l'"illumination" de l'espace de comportement
    structured_archive = None
    ...
    # =================================================================================================

    # Evaluation de la population initiale
    list(map(toolbox.evaluate, population))

    # Enregistrement de la meilleure solution et des statistiques de la population
    hof.update(population)
    stat = stats.compile(population)
    logbook.record(gen=0, pop=population, **stat)

    for g in range(1, nbgen):

        # Clonage de la population
        offspring = list(map(toolbox.clone, population))

        # Croisement et mutation
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        # ======================================================
        # PARTIE 3.3: sera à compléter plus tard
        # Ajout des nouveaux individus à l'archive structurée
        ...
        # ======================================================

        population += offspring

        # Evaluation de la nouvelle population
        list(map(toolbox.evaluate, population))

        # Sélection de la population
        population = toolbox.select(population, n)

        # Enregistrement de la meilleure solution et des statistiques de la population
        hof.update(population)
        stat = stats.compile(population)
        logbook.record(gen=g, **stat)

    return population, hof, logbook, structured_archive

Vous pouvez tester l'algorithme élitiste avec votre fonction d'évaluation avec les cellules suivantes, l'algorithme ne devrait pas prendre plus d'une minute.

Pour la suite du TME, nous allons appeler cette variante FIT, vous allez implémenter d'autres algorithmes que vous comparerez à FIT. 

In [ ]:
pop_size = 100  # Taille de la population
nb_gen = 200  # Nombre de générations

# Lancement de l'algorithme évolutionnaire
population, hof, logbook, _ = ea_elitist(
    pop_size, nb_gen, eval_arm
)

In [ ]:
# Affichage de l'évolution de la fitness
plt.plot(logbook.select("gen"), logbook.select("min"))
plt.xlabel("Generation")
plt.ylabel("Distance to target")
plt.show()

In [ ]:
# Affichage de la meilleure solution trouvée
env.fw_kinematics(hof[0])
env.display_configuration()
print("Distance to target: ", eval_arm(hof[0]))

### 2.2. NS: Optimisation mono-objectif de la nouveauté

Vous allez maintenant implémenter une nouvelle variante, consistant non plus à minimiser la distance par rapport au point cible, mais à maximiser la nouveauté des individus.

La nouveauté est mesurée dans un espace de comportement construit à la main. On projette le phénotype de chaque individu dans cet espace, et sa nouveauté est estimée en fonction de sa distance moyenne aux individus les plus proches enregistrés dans une "archive". Pour implémenter cet algorithme, il va donc falloir:

1) Implémenter la projection dans l'espace de comportement.
2) Implémenter l'archive et le score de nouveauté.

#### 2.2.1. Espace de comportement

Dans notre environnement, un bon espace de comportement est la position de l'effecteur terminal. Optimiser la nouveauté dans cet espace de comportement devrait conduire à découvrir des configurations du bras atteignant un maximum de positions diverses. 

Cet espace est décrit par deux descripteurs de comportements (*behavioral descriptors* en anglais, abrégé ici en "bd"), la position en x et en y de l'effecteur terminal. 

#### Question 2. 

Complétez la fonction suivante retournant les deux descripteurs de comportement d'un individu `ind`. Attention, en cas de collision, on renverra la position [0, 0].

In [ ]:
def compute_bd(ind):
    """
    Fonction de projection de l'individu dans l'espace de comportement.
    """
    # Initialisation de l'environnement
    env = arm.Arm(
        lengths=lengths,
        walls=walls,
        target_pos=target_pos
    )

    # A compléter
    ...

    return bd

In [ ]:
# Test de la fonction
ind1 = [0.1] * 10
ind2 = [0.2] * 10

print(compute_bd(ind1))  # Doit afficher [-1.5       9.47062727]
print(compute_bd(ind2))  # Doit afficher [0, 0]

#### 2.2.2. Archive et nouveauté

La fonction `compute_bd` permet de calculer les descripteurs de comportement associés à un individu. Dans cette partie, nous allons l'utiliser afin de calculer la nouveauté d'un individu donné.

La nouveauté d'un individu est définie comme la distance moyenne aux $k$-plus proches voisins dans l'espace de comportement (ici un plan 2D donc). Les voisins à considérer sont des individus qu'on aura au préalable sauvegardés dans une "archive" d'individus déjà évalués au cours de la recherche pas algorithme évolutionnaire.

La cellule suivante permet de définir la classe `NoveltyArchive` servant à stocker certains individus passés, servant à mesurer la nouveauté d'un nouvel individu. 


#### Question 3. 

Complétez la définition de la méthode `get_novely` permettant de calculer la nouveauté d'un individu donné à partir de l'archive.

In [ ]:
from scipy.spatial import KDTree


class NoveltyArchive:

    def __init__(self, inds, bds, k=15):
        """
        Classe pour l'archive de nouveauté.
        :param inds: liste des individus
        :param bds: liste des comportements des individus
        :param k: nombre de voisins à considérer pour le calcul de la nouveauté
        """
        self.inds = inds
        self.bds = bds
        self.k = k

        # Cette structure de données permet de trouver rapidement les distances aux k
        # plus proches voisins d'un vecteur v en utilisant l'instruction kdtree.query(v, k)
        # https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.KDTree.query.html
        self.kdtree = KDTree(np.array(bds))

    def add(self, new_inds, new_bds):
        """
        Ajoute des nouveaux individus à l'archive.
        :param new_bds: liste des comportements des individus
        """
        self.inds += new_inds
        self.bds += new_bds
        self.kdtree = KDTree(self.bds)

    def get_novelty(self, bd):
        """
        Calcule la nouveauté d'un individu.
        :param bd: comportement de l'individu
        :return: nouveauté de l'individu
        """
        # A compléter en utilisant la structure de données KDTree
        # https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.KDTree.query.html

        ...

        return novelty

La cellule suivante vous permet de tester votre implémentation:

In [ ]:
# Test de l'archive de nouveauté
bd1 = [0.0, 0.0]
bd2 = [0.0, 1.0]
bd3 = [0.0, 0.5]
bd4 = [0.0, 0.75]

archive = NoveltyArchive([None]*3, [bd1, bd2, bd3], k=2)
print(archive.get_novelty(bd1))  # Doit afficher 0.25
print(archive.get_novelty(bd4))  # Doit afficher 0.25

#### 2.2.3. Algorithme de Novelty Search (NS)

Grâce à l'espace de comportements et à l'archive permettant de calculer la nouveauté, nous pouvons maintenant implémenter la variante Novelty Search (NS). Cette variante consiste à optimiser directement la nouveauté des individus, sans considérer la distance par rapport au point cible.

Cela revient à faire un algorithme élitiste classique, en utilisant la nouveauté comme fonction d'évaluation des individus à sélectionner. Il faut également ajouter l'initialisation de l'archive permettant de calculer la nouveauté, et sa mise à jour avec de nouveaux individus sélectionnés aléatoirement dans les descendants à chaque génération.

L'algorithme de novelty search cherche à maximiser le score de nouveauté. On définit donc des nouvelles classes pour la fitness et les individus:

In [ ]:
creator.create("FitnessMax", base.Fitness, weights=(1.0,))  # Maximisation
creator.create("IndividualMax", list, fitness=creator.FitnessMax)

#### Question 4.

Complétez la cellule suivante donnant l'implémentation de la variante NS:

In [ ]:
def ea_novelty(n, nbgen, k, _lambda):
    """
    Algorithme evolutionniste avec sélection sur la nouveauté
    :param n: taille de la population
    :param nbgen: nombre de generation
    :param k: nombre de voisins à considérer pour le calcul de la nouveauté
    :param _lambda: nombre d'individus à ajouter à l'archive à chaque génération
    """
    random.seed()

    # Enregistrement des fonctions de création d'individus, de population, de sélection et d'évaluation dans le toolbox
    toolbox.register("individual", tools.initRepeat, creator.IndividualMax,
                     toolbox.attribute, n=IND_SIZE)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("select", tools.selBest)

    # Enregistrement des statistiques pour suivre l'évolution de la population
    stats = tools.Statistics(key=lambda ind: ind.fitness.values)
    stats.register("max", np.max)

    # Enregistrement de l'historique de l'évolution de la population et de la meilleure solution trouvée
    logbook = tools.Logbook()
    hof = tools.HallOfFame(1)

    # Initialisation de la population
    population = toolbox.population(n=n)

    # =================================================================================================
    # PARTIE 3.3: sera à compléter plus tard, pour l'instant vous pouvez laisser la variable à None
    # Initialisation de l'archive structurée pour étudier l'"illumination" de l'espace de comportement
    structured_archive = None
    ...
    # =================================================================================================

    # Initialisation de l'archive de nouveauté à compléter
    # On initialise l'archive avec k individus aléatoires de la population,
    # en calculant au préalable leurs descripteurs de comportement
    ...

    # Définition de la fonction d'évaluation
    # Doit calculer la nouveauté de l'individu à partir de l'archive
    def evaluate(individual):

        bd = ...
        novelty = ...

        individual.bd = bd
        individual.fitness.values = (novelty,)

    toolbox.register("evaluate", evaluate)

    # Evaluation de la population initiale
    list(map(toolbox.evaluate, population))

    hof.update(population)
    stat = stats.compile(population)
    logbook.record(gen=0, pop=population, **stat)

    for g in range(1, nbgen):

        # Clonage de la population
        offspring = list(map(toolbox.clone, population))

        # Croisement et mutation
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        # ======================================================
        # PARTIE 3.3: sera à compléter plus tard
        # Ajout des nouveaux individus à l'archive structurée
        ...
        # ======================================================

        # Mise à jour de l'archive de nouveauté
        # On y ajoute _lambda individus aléatoires parmis la population
        ...

        # Ajout des nouveaux individus à la population
        population += offspring

        # Réévaluation de toute la population (l'archive ayant été mise à jour, ça change la nouveauté)
        list(map(toolbox.evaluate, population))

        # Sélection de la population
        population = toolbox.select(population, n)

        hof.update(population)
        stat = stats.compile(population)
        logbook.record(gen=g, **stat)

    return population, hof, logbook, novelty_archive, structured_archive

Les cellules suivantes vous permettent de lancer votre variante et d'afficher les individus composant l'archive après apprentissage. L'algorithme ne devrait pas prendre plus d'une minute.

In [ ]:
pop_size = 100  # Taille de la population
nb_gen = 200  # Nombre de générations

# Lancement de l'algorithme évolutionnaire
population, hof, logbook, novelty_archive, _ = ea_novelty(
    pop_size, nb_gen, k=15, _lambda=6
)

In [ ]:
# Affichage de l'évolution de la fitness
plt.plot(logbook.select("gen"), logbook.select("max"))
plt.yscale('log')
plt.xlabel("Generation")
plt.ylabel("Novelty")
plt.show()

In [ ]:
# Affichage de la couverture de l'espace de comportement
archive_bds = np.array(novelty_archive.bds)
plt.figure(figsize=(8, 8))
plt.scatter(
    archive_bds[:, 0],
    archive_bds[:, 1],
    c='blue', alpha=0.3, label="Novelty Archive"
)
for i in range(len(walls)):
    plt.plot(walls[i, :, 0], walls[i, :, 1], color="black", linewidth=2)
plt.xlim(-15, 15)
plt.ylim(-15, 15)
plt.show()

La cellule suivante cherche dans l'archive de nouveauté la solution se rapprochant le plus de la position cible et l'affiche.

In [ ]:
distances = list(map(lambda ind: eval_arm(ind)[0], novelty_archive.inds))
amin = np.argmin(distances)
print("Distance to target: ", distances[amin])

# Affichage de la meilleure solution trouvée
env.fw_kinematics(novelty_archive.inds[amin])
env.display_configuration()

#### Question 5.

1) Cette solution est elle meilleure que la meilleure solution obtenue avec la variante FIT ? Si oui, comment l'expliquez vous, sachant que l'algorithme FIT optimise ce critère directement alors que l'algorithme NS ne l'optimise pas ?

2) Quelle est la tendance de l'évolution du score de nouveauté au cours du temps ? Comment expliquer cette tendance alors que l'algorithme évolutionnaire est configuré pour maximiser ce score ?

3) Comment expliquer la forme de l'espace de comportement couvert par les individus dans l'archive de nouveauté ?

Votre réponse ICI

### 2.3. Variante FIT+NS

Cette troisième variante combine simplement la recherche de nouveauté avec l'objectif de minimisation de la distance au point cible. Comme nous avons désormais deux objectifs, nous utilisons l'algorithme NSGA2 pour la sélection.

Nous avons un problème de minimisation/maximisation:
- Minimisation de la distance au point cible
- Maximisation de la nouveauté

On définit dans la cellule suivante les nouvelles classe de fitness et d'individu.

In [ ]:
creator.create("FitnessMinMax", base.Fitness, weights=(-1.0, 1.0))
creator.create("IndividualMinMax", list, fitness=creator.FitnessMinMax)

#### Question 6.

Complétez la fonction suivante implémentant la variante FIT+NS:

In [ ]:
def ea_fit_ns(n, nbgen, evaluate, k, _lambda):
    """
    Algorithme evolutionniste multiobjectifs avec sélection sur la nouveauté et sur la fitness
    :param n: taille de la population
    :param nbgen: nombre de generation
    :param evaluate: la fonction d'évaluation
    :param k: nombre de voisins à considérer pour le calcul de la nouveauté
    :param _lambda: nombre d'individus à ajouter à l'archive à chaque génération
    """
    random.seed()

    # Enregistrement des fonctions de création d'individus, de population, de sélection et d'évaluation dans le toolbox
    toolbox.register("individual", tools.initRepeat, creator.IndividualMinMax,
                     toolbox.attribute, n=IND_SIZE)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("select", tools.selNSGA2)

    # Initialisation de la population
    population = toolbox.population(n=n)

    # =================================================================================================
    # PARTIE 3.3: sera à compléter plus tard, pour l'instant vous pouvez laisser la variable à None
    # Initialisation de l'archive structurée pour étudier l'"illumination" de l'espace de comportement
    structured_archive = None
    ...
    # =================================================================================================

    # Initialisation de l'archive de nouveauté
    # Comme dans la variante NS, vous pouvez recopier votre code
    ...

    # Définition de la fonction d'évaluation, cette fonction d'évaluation fera deux choses:
    # 1. Calculer la distance au point cible
    # 2. Calculer la nouveauté de l'individu à partir de l'archive
    # Il pourra également être utile de sauvegarder les descripteurs de comportement dans un attribut "bd" de l'individu
    # On pourra ensuite utiliser cet attribut sans recalculer les descripteurs de comportement
    def wrapper_evaluate(individual):

        distance = ...
        bd = ...
        novelty =...

        individual.bd = bd
        individual.fitness.values = (distance, novelty)

    toolbox.register("evaluate", wrapper_evaluate)

    # Evaluation de la population initiale
    list(map(toolbox.evaluate, population))

    # Enregistrement du front de Pareto
    pareto = tools.ParetoFront()

    for g in range(1, nbgen):

        # Clonage de la population
        offspring = list(map(toolbox.clone, population))

        # Croisement et mutation
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        # ======================================================
        # PARTIE 3.3: sera à compléter plus tard
        # Ajout des nouveaux individus à l'archive structurée
        ...
        # ======================================================

        # Mise à jour de l'archive de nouveauté
        # Comme dans la variante NS, vous pouvez recopier votre code
        ...

        # Ajout des nouveaux individus à la population
        population += offspring

        # Réévaluation de toute la population (l'archive ayant été mise à jour, ça change la nouveauté)
        list(map(toolbox.evaluate, population))

        # Sélection de la population
        population = toolbox.select(population, n)

        pareto.update(population)

    return population, pareto, novelty_archive, structured_archive

Les cellules suivantes vous permettent de lancer votre variante et d'afficher les individus composant l'archive après apprentissage. L'algorithme ne devrait pas prendre plus de deux minutes.

In [ ]:
pop_size = 100  # Taille de la population
nb_gen = 200  # Nombre de générations

# Lancement de l'algorithme évolutionnaire
population, pareto, ns_fit_archive, _ = ea_fit_ns(
    pop_size, nb_gen, evaluate=eval_arm, k=15, _lambda=6
)

In [ ]:
# Affichage de la couverture de l'espace de comportement
archive_bds = np.array(ns_fit_archive.bds)
plt.figure(figsize=(8, 8))
plt.scatter(
    archive_bds[:, 0],
    archive_bds[:, 1],
    c='blue', alpha=0.3, label="Novelty Archive"
)
for i in range(len(walls)):
    plt.plot(walls[i, :, 0], walls[i, :, 1], color="black", linewidth=2)
plt.xlim(-15, 15)
plt.ylim(-15, 15)
plt.show()

La cellule suivante cherche dans l'archive de nouveauté la solution se rapprochant le plus de la position cible et l'affiche.

In [ ]:
distances = list(map(lambda ind: eval_arm(ind)[0], ns_fit_archive.inds))
amin = np.argmin(distances)
print("Distance to target: ", distances[amin])

# Affichage de la meilleure solution trouvée
env.fw_kinematics(ns_fit_archive.inds[amin])
env.display_configuration()

#### Question 7.

1) Décrivez les différences entre la couverture de l'espace de comportement dans les variantes NS et NS+FIT. Comment expliquer ces différences ?
2) La meilleure solution dans l'archive NS+FIT est-elle meilleure que la meilleure solution dans l'archive NS ? Si oui, comment expliquer ce résultat ?

Votre réponse ICI

## 3. Algorithmes de qualité-diversité

L'ensemble des solutions générées avec la variante NS peut être utilisé pour atteindre n'importe lequel des comportements atteignables, mais l'inconvénient de cette approche est que la notion de qualité est totalement absente du processus, or parmi les solutions générant un comportement donné, toutes ne se valent pas. Certaines sont plus intéressantes que d'autres parce qu'elle consomment moins d'énergie, qu'elles ne créent pas de collision entre les segments du bras, qu'elles sont plus stables, etc.

Dans cette partie, nous allons étudier deux algorithmes permettant de découvrir des solutions à la fois diverses et performantes, qu'on appelle en général algorithmes de qualité-diversité (QD):
- L'algorithme Novelty Search and Local Competition (NSLC) dans la partie 3.1.
- L'algorithme MAP-Elites dans la partie 3.2.

#### Question 8.

Pour commencer, définissez une nouvelle fonction d'évaluation `eval_std` qui n'évalue non plus la qualité d'un individu en fonction de sa distance au point cible, mais par l'écart-type de ses positions angulaires.

La fonction devra retourner l'opposé de l'écart-type des positions angulaires de l'individu (c'est directement son génotype) s'il n'y a pas de collision, et moins l'infini sinon:

In [ ]:
def eval_std(ind):
    env = arm.Arm(
        lengths=lengths,
        walls=walls,
        target_pos=target_pos
    )

    # A compléter
    ...

    return fitness,

In [ ]:
ind1 = [0.1] * 10
ind2 = [0.1] * 5 + [-0.1] * 5

print(eval_std(ind1))  # Doit afficher (np.float(0.0,))
print(eval_std(ind2))  # Doit afficher (np.float(0.10000000000000002,))

### 3.1. Variante Novelty Search and Local Competition (NSLC)

Une solution au problème de qualité-diversité consiste à utiliser, à côté de l'objectif de nouveauté, un objectif de performance. Définir cet objectif comme une pression globale est contreproductif (comme dans FIT+NS), car pour éviter des collisions ou minimiser la consommation d'énergie, il suffit de ne pas bouger... Pour rendre cette pression plus intéressante, il faut en faire un objectif non pas global, mais local.

Pour cela, on peut suivre l'approche proposée par Lehman et Stanley [1]: on compare la fitness de l'individu considéré avec celle de ses plus proches voisins (qui sont déjà déterminés pour le calcul de nouveauté). On ajoute alors un objectif de compétition locale qui vaut le nombre de voisins dont la fitness est inférieure.

* [1] Lehman, J., & Stanley, K. O. (2011). Evolving a diversity of virtual creatures through novelty search and local competition. In Proceedings of GECCO

#### 3.1.1. Archive NSLC

Nous devons commencer par modifier l'archive de nouveauté pour qu'elle permette de calculer le score de compétition locale en plus du score de nouveauté. 

#### Question 9.

Complétez la définition de la classe `NSLCArchive` définissant cette nouvelle archive. Vous pourrez vous inspirer de l'archive de nouveauté `NoveltyArchive`. Cette nouvelle archive doit à la fois calculer:

- la nouveauté, définie comme la distance moyenne aux $k$ plus proches voisins
- la qualité locale, définie comme le nombre de voisins moins performant que l'individu, divisé par $k$

In [ ]:
from scipy.spatial import KDTree


class NSLCArchive:

    def __init__(self, inds, bds, fits, k=15):
        """
        Classe pour l'archive de nouveauté et compétition locale.
        :param inds: liste des individus dans l'archive
        :param bds: liste des comportements des individus dans l'archive
        :param fits: liste des fitness des individus dans l'archive
        :param k: nombre de voisins à considérer pour le calcul de la nouveauté
        """
        self.inds = inds
        self.bds = bds
        self.fits = fits
        self.k = k

        # Cette structure de données permet de trouver rapidement les distances aux k
        # plus proches voisins d'un vecteur v en utilisant l'instruction kdtree.query(v, k)
        # https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.KDTree.query.html
        self.kdtree = KDTree(np.array(bds))

    def add(self, new_inds, new_bds, new_fits):
        """
        Ajoute des nouveaux individus à l'archive.
        :param new_inds: liste des individus à ajouter
        :param new_bds: liste des comportements des individus à ajouter
        :param new_fits: liste des fitness des individus à ajouter
        """
        self.inds += new_inds
        self.bds += new_bds
        self.fits += new_fits
        self.kdtree = KDTree(self.bds)

    def get_scores(self, bd, fit):
        """
        Calcule la nouveauté et le score de compétition locale d'un individu.
        :param bd: comportement de l'individu
        :param fitness: fitness de l'individu
        :return: nouveauté de l'individu, score de compétition locale
        """
        # A compléter en utilisant la structure de données KDTree
        # https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.KDTree.query.html

        ...

        return novelty, lc  # lc = local competition

La cellule suivante vous permet de tester votre implémentation de l'archive NSLC:

In [ ]:
# Test de l'archive de nouveauté et compétition locale
bd1 = [0.0, 0.0]
bd2 = [0.0, 2.0]
bd3 = [0.0, 1.0]

fit1 = 0.3
fit2 = 0.1
fit3 = 0.2

archive = NSLCArchive([None]*2, [bd1, bd2], [fit1, fit2], k=2)
print(archive.get_scores(bd3, fit3))  # Doit afficher (np.float(1.0), 0.5)

#### 3.1.2. Algorithme NSLC

Nous pouvons maintenant implémenter l'algorithme NSLC. Cet algorithme est très similaire à FIT+NS, car il utilise NSGA2 pour sélectionner sur deux objectifs. La seule différence est que FIT+NS utilise la fitness et la nouveauté, alors que NSLC utilise la nouveauté et le score de compétition locale.

Nous avons un problème de double maximisation:
- maximisation du score de nouveauté
- maximisation du score de compétition locale

On définit dans la cellule suivante les nouvelles classes de fitness et d'invidu pour ce problème.

In [ ]:
creator.create("FitnessMaxMax", base.Fitness, weights=(1.0, 1.0))
creator.create("IndividualMaxMax", list, fitness=creator.FitnessMaxMax)

#### Question 10.

Complétez la fonction suivant donnant l'implémentation de l'algorithme NSLC:

In [ ]:
def ea_nslc(n, nbgen, evaluate, k, _lambda):
    """
    Algorithme Novelty Search avec compétition locale
    :param n: taille de la population
    :param nbgen: nombre de generation
    :param evaluate: la fonction d'évaluation
    :param k: nombre de voisins à considérer pour le calcul de la nouveauté et de la compétition locale
    :param _lambda: nombre d'individus à ajouter à l'archive à chaque génération
    """
    random.seed()

    # Enregistrement des fonctions de création d'individus, de population, de sélection et d'évaluation dans le toolbox
    toolbox.register("individual", tools.initRepeat, creator.IndividualMaxMax,
                     toolbox.attribute, n=IND_SIZE)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("select", tools.selNSGA2)

    # Initialisation de la population
    population = toolbox.population(n=n)
    for ind in population:
        ind.fitness.values = (-np.inf, -np.inf)

    # =================================================================================================
    # PARTIE 3.3: sera à compléter plus tard, pour l'instant vous pouvez laisser la variable à None
    # Initialisation de l'archive structurée pour étudier l'"illumination" de l'espace de comportement
    structured_archive = None
    ...
    # =================================================================================================

    # Initialisation de l'archive NSLC
    # On initialise l'archive avec _lambda individus aléatoires de la population,
    # en calculant au préalable leurs descripteurs de comportement et leur fitness
    ...

    # Définition de la fonction d'évaluation
    # Doit calculer la nouveauté et le score de compétition locale de l'individu à partir de l'archive
    # Il pourra également être utile de sauvegarder les descripteurs de comportement et la fitness
    # dans des attributs "bd" et "absolute_fitness" de l'individu
    # On pourra ensuite utiliser ces attributs sans les recalculer
    def wrapper_evaluate(individual):

        bd = ...
        fitness = ...
        novelty, lc = ...

        individual.bd = bd
        individual.absolute_fitness = fitness
        individual.fitness.values = novelty, lc

    toolbox.register("evaluate", wrapper_evaluate)

    # Evaluation de la population initiale
    list(map(toolbox.evaluate, population))

    # Enregistrement du front de Pareto
    pareto = tools.ParetoFront()

    for g in range(1, nbgen):

        # Clonage de la population
        offspring = list(map(toolbox.clone, population))

        # Croisement et mutation
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        # ======================================================
        # PARTIE 3.3: sera à compléter plus tard
        # Ajout des nouveaux individus à l'archive structurée
        ...
        # ======================================================

        # Sélection des individus à ajouter à l'archive
        # On choisit _lambda individus aléatoires parmis la population
        ...

        # Ajout des nouveaux individus à la population
        population += offspring

        # Réévaluation de toute la population (l'archive ayant été mise à jour,
        # ça change la nouveauté et le score de compétition locale)
        list(map(toolbox.evaluate, population))

        # Sélection de la population
        population = toolbox.select(population, n)

        pareto.update(population)

    return population, pareto, nslc_archive, structured_archive

Les cellules suivantes vous permettent de lancer votre variante et d'afficher l'archive après apprentissage. L'algorithme ne devrait pas prendre plus de trois minutes.

In [ ]:
pop_size = 100  # Taille de la population
nb_gen = 100  # Nombre de générations

# Lancement de l'algorithme évolutionnaire
population, hof, nslc_archive, _ = ea_nslc(
    pop_size, nb_gen, evaluate=eval_std, k=15, _lambda=6
)

La cellule suivante permet d'afficher la couverture de l'espace de comportement des individus enregistrés dans l'archive NSLC, et de les colorer en fonction de leur fitness globale:

In [ ]:
archive_bds = np.array(nslc_archive.bds)
archive_fits = np.array(nslc_archive.fits)
plt.figure(figsize=(10, 8))
plt.scatter(
    archive_bds[:, 0],
    archive_bds[:, 1],
    c=archive_fits,
    alpha=1.0, label="NSLC Archive",
    cmap="viridis_r"
)
plt.colorbar(label="Fitness")
plt.xlim(-15, 15)
plt.ylim(-15, 15)
plt.show()

### 3.2. Variante MAP-Elites

#### 3.2.1. "Illuminer" l'espace des comportements

Les algorithmes QD sont aussi parfois appelés algorithmes d'illumination de l'espace des comportements. Ce surnom se réfère à l'obtention à la fois d'une couverture large de l'espace de comportement, avec des solutions performantes (tracées avec une couleur claire, comme dans la figure précédente, d'où le nom d'illumination).

Le dernier algorithme que nous allons implémenter utilise un type d'archive un peu différent. Cette archive structurée est composée de différentes cellules réalisant un pavage de l'espace de comportement. Dans notre cas, comme l'espace de comportement est en 2D, cette archive prendra la forme d'une grille de cellules en 2D.

L'algorithme MAP-Elites [1] peut être résumé comme suit:
- la grille est initialement remplie avec des individus d'une population initiale aléatoire
- à chaque génération:
    - on prend les individus de l'archive comme population parente
    - on génère des descendants de ces individus par mutation et croisement
    - on évalue la fitness et les descripteurs de comportement de ces individus
    - pour chaque nouvel individu:
        - on le compare à l'individu enregistré dans l'archive dans la cellule correspondant à ses descripteurs
        - si la cellule est vide, ou si le nouvel individu a une meilleure fitness, on assigne l'individu à cette cellule

Pour commencer l'implémentation de MAP-Elites, nous devons implémenter l'archive structurée. L'archive est composée d'une grille 2D contenant les individus enregistrés, et d'une grille 2D contenant les fitness correspondantes. Lorsqu'on ajoute des individus à la grille, il faut uniquement conserver les nouveaux individus qui ont une meilleure fitness que l'individu anciennement enregistré dans la cellule correspondante de la grille.

[1] Mouret, J. B., & Clune, J. (2015). Illuminating search spaces by mapping elites. arXiv preprint arXiv:1504.04909

#### Question 11.

Complétez la classe suivante implémentant cette archive:

In [ ]:
class StructuredArchive:

    def __init__(self, nb_cells, x_lim, y_lim):
        self.nb_cells = nb_cells
        self.x_lim = x_lim
        self.y_lim = y_lim

        # Grilles 2D contenant les individus et leur fitness
        # On peut initialiser la grille avec des individus à None et des fitness à +inf
        self.grid_ind = ...
        self.grid_fit = ...

    def add(self, new_inds, new_bds, new_fits):
        """
        Ajoute des nouveaux individus à l'archive structurée.
        :param new_inds: liste des individusà ajouter
        :param new_bds: liste des comportements des individus
        :param new_fits: liste des fitness des individus
        """
        ...

    def display(self):
        """
        Affiche l'archive de nouveauté.
        """
        masked_data = np.ma.masked_invalid(self.grid_fit.T)
        cmap = plt.cm.magma_r.copy()
        cmap.set_bad(color='black')

        plt.figure(figsize=(10, 8))
        plt.imshow(masked_data, cmap=cmap,
                   interpolation='nearest', vmin=0, vmax=2)
        plt.gca().invert_yaxis()
        plt.xticks(
            np.arange(self.nb_cells)[::5],
            self.x_lim[0] + (1/self.nb_cells) * (self.x_lim[1] -
                                                 self.x_lim[0]) * np.arange(self.nb_cells)[::5]
        )
        plt.yticks(
            np.arange(self.nb_cells)[::5],
            self.y_lim[0] + (1/self.nb_cells) * (self.y_lim[1] -
                                                 self.y_lim[0]) * np.arange(self.nb_cells)[::5]
        )
        plt.colorbar(label="Fitness")
        plt.title("Structured Archive")
        plt.show()

In [ ]:
# Test de l'archive structurée
archive = StructuredArchive(20, [-15, 15], [-15, 15])

new_inds = [None] * 3
new_bds = [[0, 0], [2, 2], [2, 2]]
new_fits = [1.5, 1, 0.5]  # L'individu 3 doit remplacer l'individu 2
archive.add(new_inds, new_bds, new_fits)

print(archive.grid_fit[11, 11])  # Doit afficher 0.5
print(archive.grid_fit[10, 10])  # Doit afficher 1.5
print(archive.grid_fit[0, 0])  # Doit afficher inf
archive.display()  # Doit afficher une image avec un pixel à (0, 0) et un pixel à (1, 1)

#### 3.2.2. Implémentation de MAP-Elites

L'archive structurée définie à la question précédente permet d'implémenter facilement l'algorithme MAP-Elites. 

Contrairement aux autres algorithmes implémentés jusqu'ici, nous n'allons pas avoir besoin de créer une population, l'archive structurée *est* la population dans MAP-Elites. Les descendants (offspring) sont générés à partir des individus de la grille, et sont ajoutés à la grille s'ils permettent de mieux illuminer l'espace des comportements (avec la méthode `add` définie précédemment).

#### Question 12.

Complétez l'implémentation de l'algorithme MAP Elites:

In [ ]:
def map_elites(nbgen, evaluate, nb_cells, x_lim, y_lim):
    """
    Algorithme Map Elites
    :param nbgen: nombre de generation
    :param evaluate: la fonction d'évaluation
    :param nb_cells: nombre de cellules dans la grille
    :param x_lim: limites de la grille en x
    :param y_lim: limites de la grille en y
    """
    random.seed()

    # Enregistrement des fonctions de création d'individus, de population et d'évaluation dans le toolbox
    toolbox.register("individual", tools.initRepeat, creator.IndividualMin,
                     toolbox.attribute, n=IND_SIZE)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate)

    # Initialisation de la population
    population = toolbox.population(n=nb_cells**2)

    # Evaluation de la population initiale
    fits = list(map(lambda ind: toolbox.evaluate(ind)[0], population))
    bds = list(map(compute_bd, population))

    # Initialisation de l'archive structurée
    # On initialise l'archive avec tous les individus de la population
    ...

    for _ in range(1, nbgen):

        # Clonage de la population
        # On clone les individus de l'archive (s'ils sont différents de None)
        ...

        # Croisement et mutation
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)

        # Ajout des nouveaux individus à l'archive structurée,
        # en calculant au préalable leurs descripteurs de comportement et leur fitness
       ...

    return structured_archive

Les cellules suivantes vous permettent de lancer votre variante et d'afficher l'archive. L'algorithme ne devrait pas prendre plus d'une minute.

In [ ]:
x_lim = [-15, 15]  # Limites de la grille en x
y_lim = [-15, 15]  # Limites de la grille en y
nb_cells = 20  # Taille de la grille (donc nb_cells x nb_cells)

# Pour la fin du TME, on passe à 50 générations
nb_gen = 50  # Nombre de générations

# Lancement de l'algorithme évolutionnaire
map_elites_archive = map_elites(
    nb_gen, evaluate=eval_std, nb_cells=nb_cells, x_lim=x_lim, y_lim=y_lim
)

La cellule suivante permet d'afficher l'archive structurée obtenue:

In [ ]:
map_elites_archive.display()

### 3.3. Comparaison des différentes variantes

#### Question 13.

Modifiez les variantes précédentes (FIT, NS, FIT+NS, NSLC) pour qu'elles construisent, mettent à jour et retournent une archive structurée contenant les meilleurs individus de chaque cellule de l'espace de comportement.

De cette manière, il est possible de comparer équitablement les différentes variantes sur leur capacité à "illuminer" l'espace de comportement.

Une fois les variantes modifiées, lancez-les une par une pour les comparer à MAP-Elites.
- On prendra `nb_cells=20` et `pop_size=400` pour être équitable (MAP-Elites a en quelque sorte une population de taille `nb_cells**2`)
- On fera attention à bien utiliser `eval_std` comme fonction d'évaluation

Affichez ensuite les différentes archives structurées obtenues.

In [ ]:
pop_size=400

In [ ]:
# Variante fit
_, _, _, fit_structured_archive = ea_elitist(
    pop_size, nb_gen, eval_std,
)

fit_structured_archive.display()

In [ ]:
# Variante ns
_, _, _, _, novelty_structured_archive = ea_novelty(
    pop_size, nb_gen, k=15, _lambda=6
)

novelty_structured_archive.display()

In [ ]:
# Variante fit+ns
_, _, _, fit_ns_structured_archive = ea_fit_ns(
    pop_size, nb_gen, evaluate=eval_std, k=15, _lambda=6
)

fit_ns_structured_archive.display()

In [ ]:
# Variante nslc
_, _, _, nslc_structured_archive = ea_nslc(
    pop_size, nb_gen, evaluate=eval_std, k=15, _lambda=6
)

nslc_structured_archive.display()

#### Question 14.

Commentez les résultats obtenus.